Library and Visual Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

#display settings 
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 100)

#visual styles
LEGIT_CLR = '#2196F3'  
FRAUD_CLR = '#F44336'  
BG        = '#F8F9FA'

plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'axes.facecolor'   : BG,
    'figure.facecolor' : 'white',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.4,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
})

print('✅ Environment ready')

Dataset Overview

In [ ]:
DATA_PATH = r"C:\Users\aashi\Desktop\Fraud Detection System\Data\Fraud Detection Dataset.csv"
df = pd.read_csv(DATA_PATH)

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage : {df.memory_usage(deep=True).sum() / 1e6:.2f} MB\n')
print('─' * 60)
print('Column data types:')
print(df.dtypes)
print('─' * 60)

In [ ]:
print('Sample rows:')
display(df.head(5))

print('\nStatistical summary (numerical columns):')
display(df.describe())

duplicate_rows   = df.duplicated().sum()
duplicate_ids    = df['Transaction_ID'].duplicated().sum()

print(f'Duplicate rows          : {duplicate_rows:,}  ({duplicate_rows/len(df)*100:.2f}%)')
print(f'Duplicate Transaction IDs: {duplicate_ids:,}  ({duplicate_ids/len(df)*100:.2f}%)')

Missing Value Analysis

In [ ]:
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %'    : (df.isnull().mean() * 100).round(2),
    'Dtype'        : df.dtypes.astype(str)
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

display(missing_summary)

# Check if missingness correlates with fraud — for each column with missing data

print('\nFraud rate: rows WITH missing value vs. rows WITHOUT')
print('─' * 55)
for col in missing_summary.index:
    is_missing = df[col].isnull()
    fraud_missing    = df[is_missing ]['Fraudulent'].mean()
    fraud_not_missing = df[~is_missing]['Fraudulent'].mean()
    diff = abs(fraud_missing - fraud_not_missing)
    flag = '⚠️ CORRELATED' if diff > 0.01 else '✅ Random'
    print(f'{col:<38} missing_fraud={fraud_missing:.4f}  present_fraud={fraud_not_missing:.4f}  {flag}')

In [ ]:
#Visualising missing value %

fig, ax = plt.subplots(figsize=(10, 4))

bars = ax.barh(missing_summary.index, missing_summary['Missing %'],
               color='#FF7043', edgecolor='white', linewidth=1.2)

# Annotate each bar with its exact percentage
for bar, val in zip(bars, missing_summary['Missing %']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=10, fontweight='bold')

# 5% reference line — beyond this, we start questioning whether to drop
ax.axvline(5, color='gray', linestyle='--', alpha=0.6, linewidth=1, label='5% threshold')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Feature')
ax.legend()
ax.set_xlim(0, missing_summary['Missing %'].max() * 1.25)
plt.tight_layout()
plt.show()

Class Imbalance Handling 

In [ ]:
counts  = df['Fraudulent'].value_counts()
ratio   = counts[0] / counts[1]

print(f'Legitimate transactions : {counts[0]:,}  ({counts[0]/len(df)*100:.2f}%)')
print(f'Fraudulent transactions : {counts[1]:,}  ({counts[1]/len(df)*100:.2f}%)')
print(f'Imbalance ratio         : {ratio:.1f}:1  (for every 1 fraud, ~{ratio:.0f} legit)')

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Target Variable: Class Distribution', fontsize=14, fontweight='bold', y=1.02)

labels  = ['Legitimate (0)', 'Fraud (1)']
colors  = [LEGIT_CLR, FRAUD_CLR]

# Donut chart
wedges, texts, autotexts = axes[0].pie(
    counts, labels=labels, colors=colors, autopct='%1.2f%%',
    startangle=90, explode=(0, 0.06),
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontweight('bold')
axes[0].set_title('Proportion of Classes')

# Bar chart
bars = axes[1].bar(labels, counts.values, color=colors, width=0.45,
                   edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[1].set_title('Absolute Count per Class')
axes[1].set_ylabel('Number of Transactions')
axes[1].set_ylim(0, counts.max() * 1.15)

plt.tight_layout()
plt.show()

Numerical Feature Distribution

In [ ]:
fraud = df[df['Fraudulent'] == 1]
legit = df[df['Fraudulent'] == 0]

num_cols = [
    'Transaction_Amount',
    'Time_of_Transaction',
    'Previous_Fraudulent_Transactions',
    'Account_Age',
    'Number_of_Transactions_Last_24H'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    ax.hist(legit[col].dropna(), bins=40, density=True, alpha=0.50,
            color=LEGIT_CLR, label='Legitimate', edgecolor='white', linewidth=0.3)
    ax.hist(fraud[col].dropna(), bins=40, density=True, alpha=0.50,
            color=FRAUD_CLR, label='Fraud', edgecolor='white', linewidth=0.3)
    legit[col].dropna().plot.kde(ax=ax, color=LEGIT_CLR, linewidth=2.5)
    fraud[col].dropna().plot.kde(ax=ax, color=FRAUD_CLR, linewidth=2.5, linestyle='--')
    ax.set_title(col.replace('_', ' '))
    ax.legend(fontsize=8)

axes[-1].set_visible(False)  # hide the empty 6th subplot
fig.suptitle('Numerical Feature Distributions: Fraud vs. Legitimate',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print skewness — this quantifies what we see visually
print('Skewness per numerical feature:')
for col in num_cols:
    skew = df[col].skew()
    flag = '⚠️  Log-transform recommended' if abs(skew) > 1 else '✅  Acceptable'
    print(f'  {col:<40} skew = {skew:+.3f}  {flag}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Transaction Amount: Raw vs. Log-Transformed', fontsize=14, fontweight='bold')

raw_amt = df['Transaction_Amount'].dropna()
log_amt = np.log1p(raw_amt)

axes[0].hist(raw_amt, bins=60, color='#7E57C2', edgecolor='white', linewidth=0.3)
axes[0].set_title(f'Raw Amount  (skewness = {raw_amt.skew():.2f})')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count')

axes[1].hist(log_amt, bins=60, color='#26A69A', edgecolor='white', linewidth=0.3)
axes[1].set_title(f'log(1 + Amount)  (skewness = {log_amt.skew():.2f})')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("""
✅ Log-transform reduces skewness from +8.45 → ~0.5.
   Decision → Apply log1p to Transaction_Amount in Phase 2.
""")

Categorical Feature Analysis

In [ ]:
cat_cols = ['Transaction_Type', 'Device_Used', 'Location', 'Payment_Method']

print('Categorical Feature Summary')
print('═' * 60)
for col in cat_cols:
    print(f'\n📌 {col}  ({df[col].nunique()} unique values)')
    vc = df[col].value_counts()
    for val, cnt in vc.items():
        pct = cnt / df[col].notna().sum() * 100
        print(f'     {val:<25} {cnt:>6,}  ({pct:.1f}%)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    rates = (df.groupby(col)['Fraudulent']
               .mean()
               .sort_values(ascending=False)
               .reset_index())
    rates.columns = [col, 'fraud_rate']

    # Colour the highest-risk bar red, others blue
    bar_colors = [FRAUD_CLR if v == rates['fraud_rate'].max() else '#90CAF9'
                  for v in rates['fraud_rate']]

    bars = ax.bar(rates[col], rates['fraud_rate'] * 100,
                  color=bar_colors, edgecolor='white', linewidth=1.2)

    for bar, val in zip(bars, rates['fraud_rate'] * 100):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f'{val:.2f}%', ha='center', va='bottom',
                fontsize=9, fontweight='bold')

    ax.set_title(f'Fraud Rate by {col.replace("_", " ")}')
    ax.set_ylabel('Fraud Rate (%)')
    ax.set_ylim(0, rates['fraud_rate'].max() * 100 * 1.25)
    ax.tick_params(axis='x', rotation=20)

fig.suptitle('Fraud Rate by Categorical Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Fraud vs Legitimate Comparison

In [ ]:
comparison_rows = []
for col in num_cols:
    l_mean, l_med = legit[col].mean(), legit[col].median()
    f_mean, f_med = fraud[col].mean(), fraud[col].median()
    comparison_rows.append({
        'Feature'       : col,
        'Legit Mean'    : round(l_mean, 2),
        'Fraud Mean'    : round(f_mean, 2),
        'Mean Δ%'       : round((f_mean - l_mean) / (l_mean + 1e-9) * 100, 2),
        'Legit Median'  : round(l_med, 2),
        'Fraud Median'  : round(f_med, 2),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.set_index('Feature'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Time of Transaction Analysis', fontsize=14, fontweight='bold')

# Distribution comparison
axes[0].hist(legit['Time_of_Transaction'].dropna(), bins=24, alpha=0.6,
             color=LEGIT_CLR, label='Legitimate', edgecolor='white')
axes[0].hist(fraud['Time_of_Transaction'].dropna(), bins=24, alpha=0.6,
             color=FRAUD_CLR, label='Fraud', edgecolor='white')
axes[0].set_xlabel('Hour of Day (0–23)')
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Hour Distribution')
axes[0].legend()

# Fraud rate by hour
hour_fraud = df.groupby('Time_of_Transaction')['Fraudulent'].mean() * 100
bar_colors = [FRAUD_CLR if v > hour_fraud.mean() else '#90CAF9'
              for v in hour_fraud.values]
axes[1].bar(hour_fraud.index, hour_fraud.values, color=bar_colors,
            edgecolor='white', linewidth=0.5)
axes[1].axhline(hour_fraud.mean(), color='gray', linestyle='--',
                linewidth=1.5, label=f'Overall avg ({hour_fraud.mean():.2f}%)')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Hour')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Prior Fraud History & Account Age Analysis', fontsize=14, fontweight='bold')

# Fraud rate by previous fraud count
prev_rate = df.groupby('Previous_Fraudulent_Transactions')['Fraudulent'].mean() * 100
bar_colors = [FRAUD_CLR if v > prev_rate.mean() else '#90CAF9' for v in prev_rate.values]
axes[0].bar(prev_rate.index, prev_rate.values, color=bar_colors, edgecolor='white')
axes[0].axhline(prev_rate.mean(), color='gray', linestyle='--', linewidth=1.5,
                label=f'Avg ({prev_rate.mean():.2f}%)')
axes[0].set_xlabel('Number of Previous Fraudulent Transactions')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_title('Fraud Rate by Prior Fraud History')
axes[0].legend()

# Account age boxplot
df.boxplot(column='Account_Age', by='Fraudulent', ax=axes[1],
           boxprops=dict(color='#37474F'),
           medianprops=dict(color=FRAUD_CLR, linewidth=2),
           whiskerprops=dict(color='#37474F'),
           capprops=dict(color='#37474F'))
axes[1].set_title('Account Age by Fraud Status')
axes[1].set_xlabel('Fraudulent (0=Legit, 1=Fraud)')
axes[1].set_ylabel('Account Age (months)')
plt.suptitle('')

plt.tight_layout()
plt.show()

Correlation Analysis 

In [ ]:
numeric_df = df.select_dtypes(include='number').drop(columns=['User_ID'])
corr = numeric_df.corr()

# Print correlation with target, sorted by strength
target_corr = corr['Fraudulent'].drop('Fraudulent').sort_values(key=abs, ascending=False)
print('Pearson correlation with target (Fraudulent):')
for feat, val in target_corr.items():
    bar = '█' * int(abs(val) * 200)
    print(f'  {feat:<40} {val:+.4f}  {bar}')

# Heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            linecolor='white', ax=ax, annot_kws={'size': 10})
ax.set_title('Feature Correlation Matrix (lower triangle)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()